<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Activity 6</strong></h4>

<p style="font-family:Times New Roman; text-align:justify; font-size:15px">
    <b>Instruction:</b> Convert the following CNN architecture diagram into a PyTorch CNN Architecture.
</p>

<center><img src="figures/quick_draw.png" width="400px"></center>
</div>

### My thought process

Before I write any PyTorch code, I want to translate the diagram into a precise, layer-by-layer spec, since jumping straight to code from a picture is how I usually end up with shape-mismatch errors. Reading it top to bottom, here's what I see:

| # | Layer | Details |
|---|---|---|
| — | Input | $(1, 28, 28)$ — 1 channel, $28\times28$ |
| 1 | Conv1 | kernel $(3,3)$, stride 1, padding 1, out\_channels 32 |
| 2 | MaxPool1 | kernel $(2,2)$, stride 2, padding 1 |
| 3 | Conv2 | kernel $(3,3)$, stride 1, padding 1, out\_channels 64 |
| 4 | Conv3 | kernel $(3,3)$, stride 1, padding 1, out\_channels 128 |
| 5 | Conv4 | kernel $(3,3)$, stride 1, padding 1, out\_channels 256 |
| 6 | MaxPool2 | kernel $(2,2)$, stride 2, padding 0 |
| 7 | Dropout | $p=0.2$ |
| 8 | Flatten | — |
| 9 | FCN1 | in = ?, out = 1000 |
| 10 | FCN2 | in = 1000, out = 500 |
| 11 | FCN3 | in = 500, out = ? (number of classes) |

One thing I noticed: every conv/pool block in the diagram is wrapped in a `ReLu( ... )` bracket, and two of those brackets enclose **both** a conv layer *and* the max-pool layer right after it (the Conv1+MaxPool1 group, and the Conv4+MaxPool2 group). At first I wasn't sure whether that meant "apply ReLU after the pooling" instead of the usual "apply ReLU right after the conv." But then I remembered that ReLU is monotonically non-decreasing, so it actually **commutes** with max-pooling: $\text{ReLU}(\text{MaxPool}(x)) = \text{MaxPool}(\text{ReLU}(x))$. So it doesn't matter which order I pick — the numbers come out the same either way. I decided to follow the same convention I saw in the CNN demo notebook from class (`x = F.relu(conv(x)); x = pool(x)`), just to keep my code in a familiar, consistent style.

**About the `?` marks in the diagram.** I don't think these are extra unknowns I have to guess — they're exactly what the output-shape formula from the CNN demo notebook is meant to compute for me:

$$\text{Output Size} = \left\lfloor \frac{\text{input size} - \text{filter} + 2 \cdot \text{padding}}{\text{stride}} \right\rfloor + 1$$

So my plan is to reuse the `calc_out()` helper from the demo notebook and work through the network dimension by dimension, layer by layer, **before** I write the model class. That way, when I get to defining `nn.Linear` for FCN1 I'll already know exactly what its `in_features` has to be, instead of guessing a number and hitting a shape-mismatch error at runtime.

**What I can't get from the diagram alone:** the number of output classes for `FCN3`. The filename (`quick_draw.png`) makes me think this is meant for a *Quick, Draw!*-style sketch classification dataset, but the exact class count really depends on which subset of categories I end up training on — that's a property of the dataset, not something the architecture diagram tells me. So instead of guessing a number, I'm exposing it as a `num_classes` parameter on my model.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print("Torch version:", torch.__version__)

### Step 1 — Working out every spatial dimension by hand first

I'm reusing the `calc_out()` helper from the CNN demo notebook to walk the *height/width* of the feature map through every conv and pool layer. Height and width stay equal to each other the whole way through here since every kernel/stride/padding in the diagram is applied symmetrically, so I only need to track one number at each step instead of two.


In [ ]:
def calc_out(w, f, s, p):
    '''
    Calculates the output spatial size of a square feature map after a
    convolution / pooling operation.
    w: input width/height
    f: filter (kernel) size
    s: stride
    p: padding
    '''
    out = (w - f + 2 * p) // s + 1
    print(f"input={w:>3} -> output={out:>3}   (kernel={f}, stride={s}, padding={p})")
    return out

print("Input: (1, 28, 28)\n")

h = 28
print("Conv1:")
h = calc_out(h, f=3, s=1, p=1)

print("\nMaxPool1:")
h = calc_out(h, f=2, s=2, p=1)

print("\nConv2 (padding=1, kernel=3, stride=1 -> should preserve size):")
h = calc_out(h, f=3, s=1, p=1)

print("\nConv3 (same as above):")
h = calc_out(h, f=3, s=1, p=1)

print("\nConv4 (same as above):")
h = calc_out(h, f=3, s=1, p=1)

print("\nMaxPool2:")
h = calc_out(h, f=2, s=2, p=0)

print(f"\nFinal feature map: 256 channels x {h} x {h}")
flatten_size = 256 * h * h
print(f"Flatten size = 256 * {h} * {h} = {flatten_size}")

Okay, so this tells me the diagram's `?` marks resolve to:

- After **Conv1**: $32 \times 28 \times 28$ — makes sense, padding=1/kernel=3/stride=1 always preserves spatial size, which is exactly the $(32, 32, 28, 28)$ the diagram already gave me as a hint.
- After **MaxPool1**: $32 \times 15 \times 15$.
- After **Conv2 / Conv3 / Conv4**: spatial size stays $15\times15$ the whole way (each is a size-preserving $3\times3$, pad 1, stride 1 conv), only the channel count changes: $32 \to 64 \to 128 \to 256$.
- After **MaxPool2**: $256 \times 7 \times 7$.
- **Flatten**: $256 \times 7 \times 7 = 12{,}544$ features per sample — this is the number I need for `FCN1`'s `in_features`.

Now I have everything I need to actually define the model, no guesswork left.


### Step 2 — Defining the model

I'm putting this in one `nn.Module` subclass, following the same structure and coding style I saw in the `CNN` class from the MNIST demo notebook.

In [ ]:
class QuickDrawCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()

        # --- convolutional feature extractor ---
        self.conv1 = nn.Conv2d(in_channels=1,   out_channels=32,  kernel_size=(3, 3), stride=1, padding=1)
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=1)

        self.conv2 = nn.Conv2d(in_channels=32,  out_channels=64,  kernel_size=(3, 3), stride=1, padding=1)
        self.conv3 = nn.Conv2d(in_channels=64,  out_channels=128, kernel_size=(3, 3), stride=1, padding=1)
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=(3, 3), stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=2, padding=0)

        self.dropout = nn.Dropout(p=0.2)

        # --- fully connected classifier head ---
        # in_features = 256 channels * 7 * 7, which I worked out in Step 1 above
        self.fcn1 = nn.Linear(in_features=256 * 7 * 7, out_features=1000)
        self.fcn2 = nn.Linear(in_features=1000, out_features=500)
        self.fcn3 = nn.Linear(in_features=500, out_features=num_classes)

    def forward(self, x):
        # Block 1: Conv1 -> ReLU -> MaxPool1
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Block 2: Conv2 -> ReLU  |  Conv3 -> ReLU  |  Conv4 -> ReLU -> MaxPool2
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool2(x)

        # regularization + flatten
        x = self.dropout(x)
        x = x.view(x.size(0), -1)   # flatten everything except the batch dimension

        # fully connected head
        x = F.relu(self.fcn1(x))
        x = F.relu(self.fcn2(x))
        x = F.softmax(self.fcn3(x), dim=1)

        return x

### Step 3 — Instantiating the model

The diagram doesn't tell me the number of output classes, so I need to set `num_classes` myself. I'll use a placeholder value for now — I'd change this to match however many *Quick, Draw!* categories I actually end up training on.

In [ ]:
NUM_CLASSES = 10  # placeholder -- I'd change this to match my actual dataset's number of classes

model = QuickDrawCNN(num_classes=NUM_CLASSES)
model

### Step 4 — Checking my shapes with a dummy forward pass

I don't want to just trust my hand calculation from Step 1 — let me actually run a dummy batch of the exact shape shown in the diagram, $(32, 1, 28, 28)$ (`batch=32`), through the network and print the tensor shape after every stage. This should match Step 1 exactly, and if it doesn't, that tells me I made a mistake somewhere.

In [ ]:
x = torch.randn(32, 1, 28, 28)   # (batch, channels, height, width)
print("Input          :", tuple(x.shape))

x1 = F.relu(model.conv1(x))
print("After Conv1    :", tuple(x1.shape))

x2 = model.pool1(x1)
print("After MaxPool1 :", tuple(x2.shape))

x3 = F.relu(model.conv2(x2))
print("After Conv2    :", tuple(x3.shape))

x4 = F.relu(model.conv3(x3))
print("After Conv3    :", tuple(x4.shape))

x5 = F.relu(model.conv4(x4))
print("After Conv4    :", tuple(x5.shape))

x6 = model.pool2(x5)
print("After MaxPool2 :", tuple(x6.shape))

x7 = model.dropout(x6)
x8 = x7.view(x7.size(0), -1)
print("After Flatten  :", tuple(x8.shape))

# and a full forward pass through the whole model
out = model(x)
print("\nFinal output   :", tuple(out.shape), " (batch, num_classes)")
print("Each row sums to 1 (softmax):", out.sum(dim=1)[:5])

### Step 5 — Parameter count

Just like in the MNIST demo notebook, I want to list the parameter count for every learnable tensor and sum them up, as a final sanity check on my architecture.

In [ ]:
def count_parameters(model):
    total = 0
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(f"{name:<12} {str(tuple(p.shape)):<20} {p.numel():>10,}")
            total += p.numel()
    print(f"{'-'*44}")
    print(f"{'Total':<33}{total:>10,}")

count_parameters(model)

### What I found

| Layer | Output shape (C, H, W) |
|---|---|
| Input | $(1, 28, 28)$ |
| Conv1 + ReLU | $(32, 28, 28)$ |
| MaxPool1 | $(32, 15, 15)$ |
| Conv2 + ReLU | $(64, 15, 15)$ |
| Conv3 + ReLU | $(128, 15, 15)$ |
| Conv4 + ReLU | $(256, 15, 15)$ |
| MaxPool2 | $(256, 7, 7)$ |
| Dropout (p=0.2) | $(256, 7, 7)$ |
| Flatten | $(12{,}544,)$ |
| FCN1 + ReLU | $(1000,)$ |
| FCN2 + ReLU | $(500,)$ |
| FCN3 + Softmax | $(\text{num\_classes},)$ |

My dummy forward pass in Step 4 confirms every one of these shapes matches what I hand-derived in Step 1, so I'm confident `QuickDrawCNN` is a faithful, runnable PyTorch translation of the architecture diagram. The one thing I left open on purpose is `num_classes`, since that really depends on which dataset I end up training this on, not on anything the diagram itself specifies.
